# Session X: Lab — Building an Interactive Personal Budget Manager

**Goal:** Combine everything from Sessions I–IX into one complete application.

---

## Exercise 1: Define the Transaction Model

Create a `Transaction` class with:
- `__init__`: amount, category, description, is_income (default False), auto-set date
- `VALID_CATEGORIES` class attribute
- `to_dict()` and `@classmethod from_dict()` for serialization
- `__repr__` for readable output
- Type hints on all methods

### How to approach this code

A Transaction class is a blueprint for one record. `VALID_CATEGORIES` is shared class data, while `__init__` stores each object's values; `to_dict()` and `from_dict()` provide the two directions between objects and JSON-friendly dictionaries. `__repr__()` makes objects readable in debugging, and the test checks that values survive a serialization round trip.


In [2]:
from datetime import datetime

class Transaction:
    VALID_CATEGORIES = ["Food", "Transport", "Entertainment", "Housing", "Utilities", "Other"]

    def __init__(self, amount: float, category: str, description: str, is_income: bool = False):
        # TODO
        self.amount = amount
        self.category = category
        self.description = description
        self.is_income = is_income
        self.date = datetime.now().isoformat()

    def to_dict(self) -> dict:
        # TODO
        return {
            "amount": self.amount,
            "category": self.category,
            "description": self.description,
            "is_income": self.is_income,
            "date": self.date
        }
        

    @classmethod
    def from_dict(cls, data: dict) -> "Transaction":
        # TODO
        t = cls(data["amount"], data["category"], data["description"], data["is_income"])
        t.date = data['date']
        return t

    def __repr__(self) -> str:
        # TODO
        kind = 'INCOME' if self.is_income else 'EXPENSE'
        return f"[{kind}] {self.category}: ${self.amount:.2f} - {self.description}"


# Test
t1 = Transaction(25.50, "Food", "Lunch")
print(t1)
t2 = Transaction(3000, "Salary", "Monthly", is_income=True)
print(t2)
data = t1.to_dict()
t1_copy = Transaction.from_dict(data)
print(f"Round-trip: {t1.amount == t1_copy.amount}")

[EXPENSE] Food: $25.50 - Lunch
[INCOME] Salary: $3000.00 - Monthly
Round-trip: True


---
## Exercise 2: Input Validation Functions

Write:
- `get_valid_amount(prompt)` — positive number, catches ValueError
- `get_valid_category(categories)` — numbered menu, validates int input
- `get_valid_description()` — rejects empty strings

### How to approach this code

Validation first converts input text, then checks whether the converted value follows the app's rules. A retry loop lets users recover from nonnumeric amounts; separate checks reject zero, negative, or out-of-range choices. Reusable helpers keep those rules consistent across menu options.


In [ ]:
def get_valid_amount(prompt: str) -> float:
    # TODO: while True + try/except
    while True:
        try:
            value = float(input(prompt))
            if value <= 0:
                print("Error: Amount must be positive.")
                continue
            return value
        except ValueError:
            print("Error: Please enter a valid number.")

def get_valid_category(categories: list[str]) -> str:
    # TODO: numbered menu + validation
    while True:
        print("\nCategories:")
        for i, cat in enumerate(categories, 1):
            print(f"    [{i}] {cat}")
        try:
            choice = int(input("Select a category: "))
            if 1 <= choice <= len(categories):
                return categories[choice - 1]
            print(f"Error: Choose between 1 and {len(categories)}.")
        except ValueError:
            print("Error: Enter a number")
def get_valid_description() -> str:
    # TODO: reject empty
    while True:
        des = input("Please enter descriptions: ").strip()
        if des:
            return des
        print("Error: Description can't be empty.")


# Uncomment to test:
# print(get_valid_amount("Amount: $"))

1.0


---
## Exercise 3: JSON Persistence

Write `save_transactions(transactions, filepath)` and `load_transactions(filepath)`.
Handle FileNotFoundError and JSONDecodeError.

### How to approach this code

The save and load functions make in-memory transactions persist between program runs. JSON stores dictionaries and lists as text; directory creation handles a first run, while `FileNotFoundError` and `JSONDecodeError` describe different situations and should receive different messages. The test uses a temporary path to avoid changing real data.


In [ ]:
import json, os

def save_transactions(transactions: list[dict], filepath: str = "data/budget.json") -> None:
    # TODO
    try: 
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "w") as f:
            json.dump(transactions, f, indent=2)
        print(f"Save {len(transactions)} transactions.")
    except:
        print("Error: File path should include folder.")

def load_transactions(filepath: str = "data/budget.json") -> list[dict]:
    # TODO
    try: 
        with open(filepath, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print("No existing data found. Starting fresh.")
        return []
    except json.JSONDecodeError:
        print("Warning: Data file is corrupted. Starting fresh.")
        return []

# Test
test_data = [{"amount": 50, "category": "Food", "description": "Groceries", "is_income": False, "date": "2026-10-01"}]
save_transactions(test_data, "./tmp/test_budget.json")
loaded = load_transactions("./tmp/test_budget.json")
print(loaded)
print(load_transactions("/tmp/nonexistent.json"))

Save 1 transactions.
[{'amount': 50, 'category': 'Food', 'description': 'Groceries', 'is_income': False, 'date': '2026-10-01'}]
No existing data found. Starting fresh.
[]


---
## Exercise 4: CSV Export

Write `export_to_csv(transactions, filepath)` using csv.DictWriter. Handle empty list.

### How to approach this code

CSV export maps each transaction dictionary to a row with a stable set of columns. `writeheader()` labels the fields and `writerows()` writes records; the empty-list check prevents a false success message. Reading the temporary export back verifies the actual output.


In [ ]:
import csv

def export_to_csv(transactions: list[dict], filepath: str = "data/export.csv") -> None:
    # TODO
    if not transactions:
        print("Nothing to export.")
        return
    
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    fieldnames = ["date", "category", "description", "amount", "is_income"]
    with open(filepath, "w", newline='') as f:
        writer = csv.DictWriter(f, fieldnames)
        writer.writeheader()
        writer.writerows(transactions)
    print(f"Exported {len(transactions)} transactions to {filepath}")


# Test
test_data = [{"date":"2026-10-01","category":"Food","description":"Groceries","amount":50.0,"is_income":False}]
export_to_csv(test_data, "./tmp/test_export.csv")
with open("./tmp/test_export.csv") as f: print(f.read())
export_to_csv([])

Exported 1 transactions to ./tmp/test_export.csv
date,category,description,amount,is_income
2026-10-01,Food,Groceries,50.0,False

Nothing to export.


---
## Exercise 5: Summary Report

Write `generate_report(transactions)` that calculates income/expenses/balance, groups by category, shows percentages. Handle empty list and ZeroDivisionError.

### How to approach this code

The report calculates income, expenses, balance, and totals by category from the same transaction list. Generator expressions select records before summing; a dictionary accumulates category totals. The percentage calculation needs a zero-expense guard, including for an empty or income-only dataset.


In [22]:
def generate_report(transactions: list[dict]) -> None:
    # TODO
    income = 0
    expenses = 0
    
    category = {}
    if not transactions:
        print("Error: There are no transactions.")
        return
    
    for t in transactions:
        if t["is_income"]:
            income += t["amount"]
        else:
            expenses += t["amount"]
            cat = t["category"]
            category[cat] = category.get(cat, 0) + t["amount"]
    balance = income - expenses

    print("\n" + "=" * 50)
    print("         BUDGET SUMMARY REPORT")
    print("=" * 50)
    print(f"  Total Income:   ${income:>10.2f}")
    print(f"  Total Expenses: ${expenses:>10.2f}")
    print(f"  Balance:        ${balance:>10.2f}")
    print("-" * 50)
    print("  Expenses by Category:")
    for cat, amount in sorted(category.items(), key=lambda x: x[1], reverse=True):
        pct = (amount / expenses * 100) if expenses > 0 else 0
        print(f"    {cat:<20} ${amount:>8.2f}  ({pct:.1f}%)")
        print("=" * 50)
  

test_data = [
    {"amount": 3000, "category": "Salary", "description": "Monthly", "is_income": True},
    {"amount": 800, "category": "Housing", "description": "Rent", "is_income": False},
    {"amount": 150, "category": "Food", "description": "Groceries", "is_income": False},
    {"amount": 75, "category": "Food", "description": "Restaurants", "is_income": False},
    {"amount": 50, "category": "Transport", "description": "Metro", "is_income": False},
]
generate_report(test_data)
generate_report([])


         BUDGET SUMMARY REPORT
  Total Income:   $   3000.00
  Total Expenses: $   1075.00
  Balance:        $   1925.00
--------------------------------------------------
  Expenses by Category:
    Housing              $  800.00  (74.4%)
    Food                 $  225.00  (20.9%)
    Transport            $   50.00  (4.7%)
Error: There are no transactions.


---
## Exercise 6: Main Application Loop

Wire everything together with a menu: Add Income, Add Expense, View All, Report, Export CSV, Save & Quit, Quit.

### How to approach this code

The main loop coordinates the application: it displays choices and calls the relevant validation, model, reporting, export, and storage operations. Keep those implementations in their own functions so each menu branch remains easy to read. Saving before exit differs from quitting without saving and should be explicit.


In [ ]:
def main():
    transactions = load_transactions()
    while True:
        print("\n" + "=" * 40)
        print("   PERSONAL BUDGET MANAGER")
        print("=" * 40)
        print("[1] Add Income  [2] Add Expense  [3] View All")
        print("[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit")
        choice = input("Select: ").strip().lower()
        if choice == "1":
            # TODO
            amount = get_valid_amount("Enter income amount: $")
            category = get_valid_category(Transaction.VALID_CATEGORIES)
            description = get_valid_description()
            transaction = Transaction(amount, category, description, is_income=True)
            transactions.append(transaction.to_dict())
        elif choice == "2":
            # TODO
            amount = get_valid_amount("Enter expense amount: $")
            category = get_valid_category(Transaction.VALID_CATEGORIES)
            description = get_valid_description()
            transaction = Transaction(amount, category, description, is_income=False)
            transactions.append(transaction.to_dict())
        elif choice == "3":
            # TODO
            for t in transactions:
                print(Transaction.from_dict(t))
        elif choice == "4":
            # TODO
            generate_report(transactions)
        elif choice == "5":
            # TODO
            export_to_csv(transactions)
        elif choice == "6":
            # TODO: save and break
            save_transactions(transactions)
            break
        elif choice == "q":
            print("Goodbye!")
            break
        else:
            print("Invalid option.")

# Uncomment to run (Ctrl-D or the stop button ends the input):
# main()

No existing data found. Starting fresh.

   PERSONAL BUDGET MANAGER
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
Error: Please enter a valid number.
